In [ ]:
import numpy as np
import sys
 
def generate_pure_polymer(beads_per_chain=5, units_x=8, units_y=8, units_z=8,
                          padding=1.0,
                          output_file="pure_polymer.data"):
    """
    Generate a tetrahedral (diamond lattice) crosslinked polymer gel
    with no solvent, no support, and no piston.
 
    Atom types:
      1 - Crosslink bead
      2 - Chain bead
 
    Bond types:
      1 - FENE chain bond
 
    A padding gap is added on all six faces so that the outermost lattice
    sites are inset from the box walls by `padding` sigma. Without this gap
    the outermost crosslinks sit exactly at the periodic boundary; during
    the harmonic pre-relaxation (NVE, fixed box) the boundary-spanning
    bonds try to expand into the wall and cause FENE errors when full
    LJ+FENE is activated. Padding gives every atom room to relax freely.
    The box is therefore gel + 2*padding in each dimension.
 
    Parameters:
    - beads_per_chain: number of beads per chain (including the two crosslink endpoints)
    - units_x, units_y, units_z: number of diamond unit cells in x, y, z
    - padding: gap between outermost lattice sites and box walls (σ), default 1.0
    - output_file: LAMMPS data file name
    """
 
    # -----------------------------------------------------------------------
    # Lattice geometry
    # -----------------------------------------------------------------------
    bead_spacing = 1.2                          # initial bond length (σ)
    chain_length = bead_spacing * (beads_per_chain - 1)
    a = chain_length                            # diamond FCC lattice constant
 
    # Gel dimensions
    gel_x = units_x * a
    gel_y = units_y * a
    gel_z = units_z * a

    # Box = gel + padding gap on all six faces
    box_x = gel_x + 2 * padding
    box_y = gel_y + 2 * padding
    box_z = gel_z + 2 * padding
 
    # Offset lattice origin so outermost sites are inset from the walls
    offset = np.array([padding, padding, padding])
 
    particles = []
    bonds     = []
    particle_id = 1
    bond_id     = 1
    molecule_id = 1
 
    # -----------------------------------------------------------------------
    # Crosslink positions (diamond lattice)
    # Diamond = 2 interpenetrating FCC sub-lattices offset by (a/4, a/4, a/4)
    # -----------------------------------------------------------------------
    crosslinks = {}
 
    fcc_basis = [
        np.array([0.0, 0.0, 0.0]),
        np.array([0.5, 0.5, 0.0]),
        np.array([0.5, 0.0, 0.5]),
        np.array([0.0, 0.5, 0.5]),
    ]
 
    for i in range(units_x + 1):
        for j in range(units_y + 1):
            for k in range(units_z + 1):
                for sublattice in [0, 1]:
                    for fcc_idx, fcc_frac in enumerate(fcc_basis):
                        if sublattice == 1:
                            frac_pos = fcc_frac + np.array([0.25, 0.25, 0.25])
                        else:
                            frac_pos = fcc_frac
 
                        pos = (np.array([i, j, k]) + frac_pos) * a + offset

                        # Keep only sites within the gel region
                        gel_pos = pos - offset
                        if not (0 <= gel_pos[0] <= gel_x and
                                0 <= gel_pos[1] <= gel_y and
                                0 <= gel_pos[2] <= gel_z):
                            continue

                        key = tuple(np.round(pos, 6))
                        if key in crosslinks:
                            continue

                        crosslinks[key] = {
                            'id': particle_id,
                            'pos': pos.copy(),
                        }
                        particles.append({
                            'id':   particle_id,
                            'type': 1,
                            'pos':  pos.copy(),
                            'mol':  0,
                        })
                        particle_id += 1
 
    # -----------------------------------------------------------------------
    # Chain beads + bonds (nearest-neighbour crosslink pairs)
    # Nearest-neighbour distance in diamond lattice: a*sqrt(3)/4
    # -----------------------------------------------------------------------
    bond_distance = a * np.sqrt(3) / 4
    created_bonds = set()
 
    crosslink_list = list(crosslinks.values())
 
    for idx1, cl1 in enumerate(crosslink_list):
        id1   = cl1['id']
        pos1  = cl1['pos']
 
        for idx2, cl2 in enumerate(crosslink_list):
            id2  = cl2['id']
            pos2 = cl2['pos']
 
            if id1 >= id2:
                continue
 
            # Plain Euclidean distance — no PBC bonds since lattice is inset
            dr   = pos2 - pos1
            dist = np.linalg.norm(dr)
 
            if abs(dist - bond_distance) < 0.1 * bond_distance:
                bond_pair = tuple(sorted([id1, id2]))
                if bond_pair in created_bonds:
                    continue
                created_bonds.add(bond_pair)
 
                chain_ids = [id1]
 
                # Interior chain beads interpolated along the bond vector
                for b in range(1, beads_per_chain - 1):
                    frac = b / (beads_per_chain - 1)
                    pos  = pos1 + frac * dr
                    particles.append({
                        'id':   particle_id,
                        'type': 2,            # Chain bead
                        'pos':  pos.copy(),
                        'mol':  molecule_id,
                    })
                    chain_ids.append(particle_id)
                    particle_id += 1
 
                chain_ids.append(id2)
 
                # Assign molecule ID to the two crosslinks of this chain
                particles[id1 - 1]['mol'] = molecule_id
                particles[id2 - 1]['mol'] = molecule_id
 
                for b in range(len(chain_ids) - 1):
                    bonds.append({
                        'id':    bond_id,
                        'type':  1,
                        'atom1': chain_ids[b],
                        'atom2': chain_ids[b + 1],
                    })
                    bond_id += 1
 
                molecule_id += 1
 
    # -----------------------------------------------------------------------
    # Write LAMMPS data file
    # -----------------------------------------------------------------------
    with open(output_file, 'w') as f:
        f.write("LAMMPS data file: pure polymer diamond-lattice gel (no solvent)\n\n")
        f.write(f"{len(particles)} atoms\n")
        f.write(f"{len(bonds)} bonds\n")
        f.write("0 angles\n")
        f.write("0 dihedrals\n")
        f.write("0 impropers\n\n")
        f.write("2 atom types\n")
        f.write("1 bond types\n\n")
        f.write(f"0.0 {box_x:.6f} xlo xhi\n")
        f.write(f"0.0 {box_y:.6f} ylo yhi\n")
        f.write(f"0.0 {box_z:.6f} zlo zhi\n\n")
        f.write("Masses\n\n")
        f.write("1 1.0  # Crosslink\n")
        f.write("2 1.0  # Chain bead\n\n")
        f.write("Atoms\n\n")
        for p in particles:
            f.write(f"{p['id']} {p['mol']} {p['type']} "
                    f"{p['pos'][0]:.6f} {p['pos'][1]:.6f} {p['pos'][2]:.6f}\n")
        f.write("\nBonds\n\n")
        for b in bonds:
            f.write(f"{b['id']} {b['type']} {b['atom1']} {b['atom2']}\n")
 
    # -----------------------------------------------------------------------
    # Summary
    # -----------------------------------------------------------------------
    num_crosslinks = sum(1 for p in particles if p['type'] == 1)
    num_chain      = sum(1 for p in particles if p['type'] == 2)
    num_chains     = len(bonds) // (beads_per_chain - 1) if beads_per_chain > 1 else 0
 
    print("Generated pure polymer diamond-lattice gel:")
    print(f"  Unit cells:        {units_x} x {units_y} x {units_z}")
    print(f"  Beads per chain:   {beads_per_chain}  (including crosslink endpoints)")
    print(f"  Lattice constant:  {a:.4f} σ")
    print(f"  Bond distance:     {bond_distance:.4f} σ")
    print(f"  Padding:           {padding:.4f} σ  (gap on each face)")
    print(f"  Gel dimensions:    {gel_x:.4f} x {gel_y:.4f} x {gel_z:.4f} σ")
    print(f"  Box dimensions:    {box_x:.4f} x {box_y:.4f} x {box_z:.4f} σ")
    print(f"  Crosslinks:        {num_crosslinks}")
    print(f"  Chain beads:       {num_chain}")
    print(f"  Chains (bonds/4):  {num_chains}")
    print(f"  Total atoms:       {len(particles)}")
    print(f"  Total bonds:       {len(bonds)}")
    print(f"  Output:            {output_file}")


In [ ]:
# -----------------------------------------------------------------------
# Inputs
# -----------------------------------------------------------------------
numBeads    = 5
unitsXY     = 8
unitsZ      = 8
padding     = 1.0   # gap between lattice edge and box wall (σ)
outputFile  = "../../lammps_data/polymer_pure/pure_polymer_5beads_8x8x8.data"
 
generate_pure_polymer(numBeads, unitsXY, unitsXY, unitsZ, padding, outputFile)


In [ ]:
import numpy as np

P   = np.array([0.005, 0.01, 0.015, 0.02, 0.025, 0.03, 0.035, 0.04, 0.045, 0.05,
                0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0])
rho = np.array([0.922125333340782, 0.922241327565221, 0.922570304599084, 0.922992392473176,
                0.923103955766514, 0.923373688359114, 0.923477896964325, 0.923929508957248,
                0.924025900913588, 0.924261848512413,
                0.926748982281438, 0.931265628027888, 0.935722245190837, 0.939845347506881,
                0.943868045667394, 0.947763224852455, 0.951646101263352, 0.955123882401049,
                0.958803706712757, 0.96217237541906])

# Centered differences (interior), one-sided at endpoints
drho_dP       = np.gradient(rho, P)   # np.gradient handles non-uniform spacing
beta_T        = drho_dP / rho
K_p           = 1 / beta_T

print(f"{'P*':>8}  {'rho*':>8}  {'beta_T*':>12}  {'K_p*':>12}")
for p, r, b, k in zip(P, rho, beta_T, K_s):
    print(f"{p:>8.3f}  {r:>8.5f}  {b:>12.5f}  {k:>12.5f}")

    
# κ_T = 1/β is the isothermal bulk modulus. Want κ_T >> M, G for the gel    
